# 02. Survivor-Narrative Annotation Sample

This notebook draws the human-annotation sample used to develop and evaluate the survivor-narrative classifier. The sampling frame is the finalized Candidate Corpus created in Notebook 01.

In [2]:
# Import packages used for data management and sampling.
import os
import pandas as pd

# Set the project root as the working directory so that relative paths
# are consistent across project notebooks.
os.chdir("/Users/Kashif/Library/CloudStorage/Box-Box/0.KB_NLP")

print("Working directory:", os.getcwd())

Working directory: /Users/Kashif/Library/CloudStorage/Box-Box/0.KB_NLP


## Load the Candidate Corpus

The finalized Candidate Corpus contains Reddit submissions that met the predefined text-availability criteria and remained after conservative duplicate removal. This frozen corpus serves as the sampling frame for survivor-narrative annotation.

In [3]:
# Load the finalized Candidate Corpus produced in Notebook 01.
candidate = pd.read_json(
    "data/processed/reddit_candidate_corpus.jsonl",
    lines=True
)

print("Candidate corpus:", candidate.shape)

Candidate corpus: (2960, 141)


In [23]:
# Confirm the expected number of observations and unique Reddit IDs.
print("Posts:", len(candidate))
print("Unique IDs:", candidate["id"].nunique())
print("Duplicate IDs:", candidate["id"].duplicated().sum())

Posts: 2960
Unique IDs: 2960
Duplicate IDs: 0


## Draw the Annotation Sample

A simple random sample of 500 posts was drawn from the finalized Candidate Corpus for manual survivor-narrative annotation. A fixed random seed (`random_state=42`) was used to make the sampling procedure reproducible. Sampling was performed without reference to the previous pilot annotation dataset.

In [4]:
# Draw a reproducible simple random sample of 500 posts from the
# finalized Candidate Corpus.
annotation_sample = candidate.sample(
    n=500,
    random_state=42
).copy()

# Retain the original title and selftext fields as well as text_raw,
# which represents the combined textual input used for modeling.
annotation_sample = annotation_sample[
    [
        "id",
        "title",
        "selftext",
        "text_raw",
        "created_utc"
    ]
].copy()

# Create an empty variable for manual survivor-narrative coding.
annotation_sample["survivor_narrative"] = pd.NA

# Verify the resulting annotation sample.
print("Annotation sample:", annotation_sample.shape)
print("Unique IDs:", annotation_sample["id"].nunique())
print("Duplicate IDs:", annotation_sample["id"].duplicated().sum())
print("Missing text_raw:", annotation_sample["text_raw"].isna().sum())

Annotation sample: (500, 6)
Unique IDs: 500
Duplicate IDs: 0
Missing text_raw: 0


# *SCRATCH*

In [25]:
# Load previously completed annotations so that existing human coding
# can be reused when posts reappear in the definitive annotation sample.
old_ann = pd.read_excel(
    "data/annotated/zzzarchive/0.narr_ann.xlsx"
)

old_train = pd.read_excel(
    "data/annotated/zzzarchive/narr_train.xlsx"
)

print("Previous annotation sample:", old_ann.shape)
print(
    "Previous annotation labels:",
    old_ann["survivor_narrative"].notna().sum()
)

print("\nOriginal training annotations:", old_train.shape)
print(
    "Original training labels:",
    old_train["survivor_narrative"].notna().sum()
)

Previous annotation sample: (500, 6)
Previous annotation labels: 332

Original training annotations: (580, 5)
Original training labels: 580


In [26]:
# Identify overlap between the definitive 500-post sample and each
# previously annotated dataset.
new_ids = set(annotation_sample["id"])
ann_ids = set(old_ann["id"])
train_ids = set(old_train["id"])

print(
    "New sample overlapping previous annotation sample:",
    len(new_ids & ann_ids)
)

print(
    "New sample overlapping original training annotations:",
    len(new_ids & train_ids)
)

print(
    "New sample found in either annotation source:",
    len(new_ids & (ann_ids | train_ids))
)

# Check agreement where the two archived datasets contain the same post
# and both have a completed survivor-narrative label.
archive_overlap = old_ann[
    ["id", "survivor_narrative"]
].merge(
    old_train[["id", "survivor_narrative"]],
    on="id",
    how="inner",
    suffixes=("_ann", "_train")
)

archive_overlap_coded = archive_overlap.dropna(
    subset=["survivor_narrative_ann", "survivor_narrative_train"]
)

disagreements = archive_overlap_coded.loc[
    archive_overlap_coded["survivor_narrative_ann"]
    != archive_overlap_coded["survivor_narrative_train"]
]

print("\nPosts coded in both archives:", len(archive_overlap_coded))
print("Label disagreements:", len(disagreements))

New sample overlapping previous annotation sample: 217
New sample overlapping original training annotations: 133
New sample found in either annotation source: 268

Posts coded in both archives: 139
Label disagreements: 0


In [27]:
# Combine all previously completed human-coded labels.
# Only non-missing annotations are retained. Because the two archived
# datasets showed no disagreements, labels can be safely pooled.

prior_labels = pd.concat(
    [
        old_train[["id", "survivor_narrative"]],
        old_ann[["id", "survivor_narrative"]]
    ],
    ignore_index=True
)

# Remove rows that were never coded.
prior_labels = prior_labels.dropna(
    subset=["survivor_narrative"]
)

# If the same post appears in both archives, retain a single copy.
prior_labels = prior_labels.drop_duplicates(
    subset=["id"],
    keep="first"
)

# Map completed prior annotations onto the definitive sample.
prior_label_map = prior_labels.set_index(
    "id"
)["survivor_narrative"]

annotation_sample["survivor_narrative"] = (
    annotation_sample["id"].map(prior_label_map)
)

print(
    "Previously coded labels recovered:",
    annotation_sample["survivor_narrative"].notna().sum()
)

print(
    "Posts remaining to annotate:",
    annotation_sample["survivor_narrative"].isna().sum()
)

Previously coded labels recovered: 210
Posts remaining to annotate: 290


In [28]:
# Export the definitive 500-post annotation sample.
# Previously completed human annotations are retained, while uncoded
# observations remain missing for subsequent manual coding.
annotation_sample.to_excel(
    "data/annotated/narr_ann.xlsx",
    index=False
)

print("Saved annotation sample:", annotation_sample.shape)
print(
    "Previously coded:",
    annotation_sample["survivor_narrative"].notna().sum()
)
print(
    "Remaining to annotate:",
    annotation_sample["survivor_narrative"].isna().sum()
)

Saved annotation sample: (500, 6)
Previously coded: 210
Remaining to annotate: 290


# Addressing Class Imbalance

422 positives vs. 78 negatives (84.4% vs. 15.6%)

**Supplemental annotation sample.** Following annotation of the initial 500-post random sample, an additional simple random sample of 200 previously unsampled posts was drawn from the remaining 2,460 submissions in the Candidate Corpus using a fixed random seed (random_state=43). This supplemental sample was created to identify additional non-survivor narratives for minority-class enrichment. The complete 200-post sample was retained separately to preserve the original random draw and permit transparent documentation of subsequent class-based selection.

In [5]:
# Identify posts that were not included in the original 500-post annotation sample.
# Sampling only from these posts prevents duplicate annotation.
remaining_pool = candidate.loc[
    ~candidate["id"].isin(annotation_sample["id"])
].copy()

print("Posts available for additional sampling:", len(remaining_pool))

# Draw an additional simple random sample of 200 posts.
# Use a different fixed seed from the original sample so this draw is reproducible
# and independent of the initial random sample.
additional_sample = remaining_pool.sample(
    n=200,
    random_state=43
).copy()

# Retain the same variables used in the original annotation workbook.
additional_sample = additional_sample[
    [
        "id",
        "title",
        "selftext",
        "text_raw",
        "created_utc"
    ]
].copy()

additional_sample["survivor_narrative"] = pd.NA

print("Additional annotation sample:", additional_sample.shape)
print("Unique IDs:", additional_sample["id"].nunique())
print(
    "Overlap with original 500:",
    additional_sample["id"].isin(annotation_sample["id"]).sum()
)

Posts available for additional sampling: 2460
Additional annotation sample: (200, 6)
Unique IDs: 200
Overlap with original 500: 0


In [7]:
# Save the supplemental sample separately so the complete random draw
# is preserved before any class-based selection occurs.
additional_sample.to_excel(
    "data/annotated/class_imb1_200.xlsx",
    index=False
)